In [1]:
import pandas as pd
from pathlib import Path


In [2]:
folder_path = Path("data/excel_data")

# Read and combine all .xlsx files
files = list(folder_path.glob("*.xlsx"))

df = pd.concat([pd.read_excel(f) for f in files], ignore_index=True)


/Users/abbymihaly/.venvs/lede/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/abbymihaly/.venvs/lede/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/abbymihaly/.venvs/lede/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/abbymihaly/.venvs/lede/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/abbymihaly/.venvs/lede/lib/python3.13

In [3]:
df.head()

,RecordNumber,FileNo,BusinessType,BusinessName,NaicsCodes,PrincipalOfficeAddress1,PrincipalOfficeAddress2,PrincipalOfficeCity,PrincipalOfficeState,PrincipalOfficeZipCode,...,AgentZipCode,DateOfIncorporation,BusinessStatus,LastAnnualReportDate,TerminationDate,FiscalYearMonth,TermDate,DissolvedDate,SubType,ExpirationDate
0,22341,L0023033,Domestic Business Corporation,"VERMONT WOOD PELLET COMPANY, INC.",321999-All Other Miscellaneous Wood Product Ma...,1105 RTE 7B Central,NaN,NORTH CLARENDON,VT,05759,...,05759,2008-04-14,Active - In Good Standing,2025-12-31,NaN,12.0,0001-01-01,NaN,NaN,NaN
1,40981,V00046,Domestic Business Corporation,NEWPORT AND RICHFORD RAILROAD COMPANY,NaN,84 PINE ST,NaN,BURLINGTON,VT,05402,...,05402,1880-12-06,Inactive - Administratively Terminated,2001-12-31,NaN,12.0,NaN,NaN,NaN,NaN
2,40988,V00052,Domestic Business Corporation,CLARENDON AND PITTSFORD RAILROAD COMPANY,NaN,ONE RAILWAY LN.,NaN,BURLINGTON,VT,05401,...,05401,1885-09-09,Active - In Good Standing,2025-12-31,NaN,12.0,NaN,NaN,NaN,NaN
3,41052,V00116,Domestic Business Corporation,RUTLAND FIRE CLAY CO.,NaN,8 Madison Ave,NaN,Rutland,VT,05701,...,05464,1883-12-12,Active - In Good Standing,2025-12-31,NaN,12.0,NaN,NaN,NaN,NaN
4,41062,V00124,Domestic Business Corporation,"PGL AND SONS HOLDINGS, INC.",NaN,1118 RT 14,NaN,HARTFORD,VT,05047,...,05001,1881-04-21,Inactive - Administratively Terminated,2012-12-31,2014-06-01,12.0,NaN,NaN,NaN,NaN


### Making new df with only General Stores.

Defining general stors as those stores that include in their titles:
1) "General Store"
2) "Country Store"
or 
3) "Store" + name of a Vermont city or town

In [4]:
# first, list all vermont towns

df_towns = pd.read_csv('Vermont Town Data/FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1_-8833757079210321760.csv')

df_towns.head()

,OBJECTID,FIPS6,TOWNNAME,TOWNNAMEMC,CNTY,TOWNGEOID,Shape__Area,Shape__Length
0,1,9030.0,CANAAN,Canaan,9,5000911800,8.613393e+07,63085.366335
1,2,11040.0,FRANKLIN,Franklin,11,5001127100,1.058529e+08,42591.256754
2,3,11015.0,BERKSHIRE,Berkshire,11,5001105425,1.085458e+08,42302.494352
3,4,11050.0,HIGHGATE,Highgate,11,5001133025,1.555736e+08,57367.840529
4,5,11060.0,RICHFORD,Richford,11,5001159125,1.118973e+08,42398.623987


Now make a new df_general_stores with those three filters General Store, Country Store and town + Store

In [5]:
# Title case all the towns
df_towns['TOWNNAME'] = df_towns['TOWNNAME'].str.title()

# Make a list of towns
town_list = df_towns["TOWNNAME"].tolist()

town_list

['Canaan',
 'Franklin',
 'Berkshire',
 'Highgate',
 'Richford',
 'Alburgh',
 'Norton',
 'Averill',
 'Holland',
 'Jay',
 'Troy',
 'Lemington',
 'Isle La Motte',
 "Avery'S Gore",
 "Warren'S Gore",
 "Warner'S Grant",
 'Sheldon',
 'Morgan',
 'Lewis',
 'Swanton',
 'North Hero',
 'Enosburgh',
 'Westfield',
 'Charleston',
 'Saint Albans Town',
 'Brownington',
 'Fairfield',
 'Bloomfield',
 'Brighton',
 'Irasburg',
 'Lowell',
 'Bakersfield',
 'Saint Albans City',
 'Westmore',
 'Ferdinand',
 'Brunswick',
 'Belvidere',
 'Georgia',
 'Albany',
 'Grand Isle',
 'Waterville',
 'Fairfax',
 'Newark',
 'Fletcher',
 'Barton',
 'Sutton',
 'Glover',
 'Milton',
 'East Haven',
 'Craftsbury',
 'Cambridge',
 'Sheffield',
 'South Hero',
 'Granby',
 'Greensboro',
 'Burke',
 'Westford',
 'Johnson',
 'Wolcott',
 'Underhill',
 'Wheelock',
 'Guildhall',
 'Victory',
 'Hyde Park',
 'Morristown',
 'Hardwick',
 'Stannard',
 'Kirby',
 'Stowe',
 'Elmore',
 'Lyndon',
 'Walden',
 'Lunenburg',
 'Jericho',
 'Saint Johnsbury',


In [6]:
# for "GENERAL STORE" and "COUNTRY STORE" / add word boundaries to the standalone phrases 
phrase_mask = df["BusinessName"].str.contains(
    r"\bGeneral Store\b|\bCountry Store\b",
    case=False,
    na=False,
)

# for town + "store"
# create a pattern for each town: must contain both 'Store' and the town name.
# Enforce standalone word boundaries (\b) around the word 'store'
town_pattern = "|".join(fr"(?=.*\bstore\b)(?=.*{town})" for town in town_list)

town_mask = df["BusinessName"].str.contains(
    town_pattern,
    case=False,
    na=False,
)

# filter df
df_general_stores = df[phrase_mask | town_mask]

In [7]:
df_general_stores.head(40)

,RecordNumber,FileNo,BusinessType,BusinessName,NaicsCodes,PrincipalOfficeAddress1,PrincipalOfficeAddress2,PrincipalOfficeCity,PrincipalOfficeState,PrincipalOfficeZipCode,...,AgentZipCode,DateOfIncorporation,BusinessStatus,LastAnnualReportDate,TerminationDate,FiscalYearMonth,TermDate,DissolvedDate,SubType,ExpirationDate
753,55376,V10041,Domestic Business Corporation,"OLD BENNINGTON COUNTRY STORE, INC.",NaN,39 WEST ROAD,NaN,BENNINGTON,VT,05201,...,05201,1960-06-17,Dissolved,1985-12-31,NaN,12.0,NaN,1987-03-20,NaN,NaN
799,55857,V10258,Domestic Business Corporation,"THE VERMONT COUNTRY STORE, INC.",454110-Electronic Shopping and Mail-Order Houses,5650 Main Street,NaN,Manchester Center,VT,05255,...,05255,1961-01-09,Active - In Good Standing,2026-04-30,NaN,4.0,0001-01-01,NaN,NaN,NaN
913,57273,V10910,Domestic Business Corporation,"COUNTRY STORE RESTAURANT, INC.",NaN,NaN,NaN,MANCHESTER,VT,05254,...,05061,1962-08-02,Inactive - Administratively Terminated,1987-07-31,NaN,7.0,NaN,NaN,NaN,NaN
1053,58742,V11578,Domestic Business Corporation,"GRAFTON VILLAGE STORE, INC.",NaN,5 BROWN STREET,NaN,BELLOWS FALLS,VT,05101,...,05101,1963-12-16,Dissolved,2009-12-31,NaN,12.0,NaN,2010-03-10,NaN,NaN
1136,59684,V12009,Domestic Business Corporation,"CHARLESTOWN MILL STORE OF BENNINGTON, INC.",NaN,PO BOX F,NaN,CHARLESTOWN,NH,03603,...,05682,1964-09-22,Inactive - Administratively Terminated,1982-06-30,NaN,6.0,NaN,NaN,NaN,NaN
1542,62004,V13840,Domestic Business Corporation,"BERARDINELLI'S GENERAL STORE, INC.",NaN,MAIN ST RT 104,NaN,FAIRFAX,VT,05454,...,05444,1967-10-24,Inactive - Administratively Terminated,1993-12-31,NaN,12.0,NaN,NaN,NaN,NaN
1867,63049,V15260,Domestic Business Corporation,"HISTORIC CRAFTSBURY GENERAL STORE, INC.",NaN,MAIN ST.,NaN,CRAFTSBURY,VT,05826,...,05826,1969-09-29,Dissolved,1995-09-30,NaN,9.0,NaN,1995-11-08,NaN,NaN
2046,63967,V16003,Domestic Business Corporation,"WAITS RIVER GENERAL STORE, INC.",NaN,NaN,NaN,WAITS RIVER,VT,05086,...,05086,1970-08-25,Dissolved,1983-12-31,NaN,12.0,NaN,1985-05-09,NaN,NaN
2067,64076,V16092,Domestic Business Corporation,"WARDSBORO COUNTRY STORE, INC.",NaN,NaN,NaN,WARDSBORO,VT,05355,...,05355,1970-10-08,Dissolved,1991-12-31,NaN,12.0,NaN,1992-04-14,NaN,NaN
2072,64104,V16115,Domestic Business Corporation,"FRECHETTE'S COUNTRY STORE, INC.",NaN,NaN,NaN,NORTON,VT,05907,...,05855,1970-10-19,Dissolved,1985-12-31,NaN,12.0,NaN,1986-09-30,NaN,NaN


In [ ]:
# List of business names to exclude
exclude_names = ['DOLLAR GENERAL STORE', 'COUNTRY STORE PROPERTIES LLC', ]

# Filter out those rows
df_general_stores = df_general_stores[~df_general_stores['BusinessName'].isin(exclude_names)] 
# also need to exclude business names that are other kinds of stores that may include store + city name
df_general_stores = df_general_stores[~df_general_stores['BusinessName'].str.contains(r"\b(BOOK STORE|HARDWARE STORE|THRIFT STORE|GROCERY STORE|SOLAR STORE|FARM STORE|DRUG STORE|ARMY AND NAVY STORE|BRICK STORE|LIQUOR STORE|TOY STORE|BRICK STORE|COLLEGE|DUTY FREE|)\b")]

/var/folders/4t/dlz7f1jn6gl_h8g2l5076hj80000gn/T/ipykernel_38818/1375779940.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_general_stores = df_general_stores[~df_general_stores['BusinessName'].str.contains(r"\b(BOOK STORE|HARDWARE STORE|THRIFT STORE|GROCERY STORE|SOLAR STORE|FARM STORE|DRUG STORE|ARMY AND NAVY STORE|BRICK STORE|LIQUOR STORE|TOY STORE|BRICK STORE|COLLEGE)\b")]


In [22]:
# checking to make sure this all worked -- here is the list of stores that dont include "country" or "genera" but are rathe just the town names plus "store"
subset = df_general_stores[
    ~df_general_stores["BusinessName"].str.contains(
        r"\b(country|general)\b",
        case=False,
        na=False
    )]


/var/folders/4t/dlz7f1jn6gl_h8g2l5076hj80000gn/T/ipykernel_38818/78607812.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df_general_stores["BusinessName"].str.contains(


In [27]:
subset_list = subset['BusinessName'].unique()
for item in subset_list:
    print(item)

GRAFTON VILLAGE STORE, INC.
CHARLESTOWN MILL STORE OF BENNINGTON, INC.
MIDDLETOWN SPRINGS EAST STREET STORE, INC.
FAIRFAX VILLAGE STORE, INC.
PEACHAM STORE, INC.
CASTLETON VILLAGE STORE, INC.
GEORGIA CORNER STORE, INC.
SHELBURNE MILL STORE, INC. THE
TINMOUTH STORE, LTD., THE
NORTH POMFRET STORE, INC., THE
CABOT VILLAGE STORE, INC.
BRADFORD VILLAGE STORE, INC.
WINCHESTER'S STORE, INC.
VILLAGE STORE OF MONKTON, INC.
WESTON VILLAGE STORE, INC.
WOLCOTT STORE, INC.
TOWNSHEND CORNER STORE, INC.
GRAND ISLE STORE, INC. THE
SOUTH LONDONDERRY VILLAGE STORE, INC.
NORTON DUTY FREE STORE, INC.
SIMON'S MILTON STORE, INC.
NEWBURY VILLAGE STORE, INC.
O'BRIEN'S STORE OF WILLISTON, INC.
WEST HARTFORD VILLAGE STORE, INC.
PROCTOR STORE, INC.
SIMON'S WAITSFIELD STORE, INC.
CHITTENDEN STORE & DELI, INC
MARSHFIELD VILLAGE STORE INC
NORTH BENNINGTON VARIETY STORE INC
AR LYNDONVILLE STORE CORP
JRY Burlington Store Corporation
AR SWANTON STORE CORP.
BROOKFIELD VALLEY STORE, LLC
BENSON VILLAGE STORE, LLC
TUNBRID

In [28]:
df_general_stores.to_csv('data/cleaned/general_stores_sorted_from_excelsheets.csv', index=False)